<a href="https://colab.research.google.com/github/h186487/DAT158MLSopp/blob/main/notebooks/01b_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01b: Metrikker og baselines

Notebooken definerer metrikkene prosjektet bruker og måler ytelsen til enkle baselines på testsettet. Resultatene brukes som sammenligningsgrunnlag for modellene i modelleringsnotebooken.

## 0. Last inn data

In [ ]:
import pandas as pd

base = "https://raw.githubusercontent.com/h186487/DAT158MLSopp/main/data/processed"
train = pd.read_csv(f"{base}/train.csv")
test = pd.read_csv(f"{base}/test.csv")

Xtr, ytr = train.drop(columns="poisonous"), train["poisonous"]
Xte, yte = test.drop(columns="poisonous"), test["poisonous"]
print(Xtr.shape, Xte.shape)

(6499, 21) (1625, 21)


## 1. Metrikker

In [ ]:
from sklearn.metrics import recall_score, precision_score, accuracy_score, confusion_matrix

def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "Modell": name,
        "Recall (giftig)": round(recall_score(y_true, y_pred, zero_division=0), 3),
        "Precision (giftig)": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "Accuracy": round(accuracy_score(y_true, y_pred), 3),
        "False negatives": fn,
        "False positives": fp,
    }

Giftig er positiv klasse (1). Hovedmetrikken er **recall for giftig**: andelen giftige sopper som modellen fanger opp. En false negative (modellen sier spiselig, soppen er giftig) er den farligste feilen. Precision (andelen av "giftig"-prediksjonene som faktisk er giftige) og false positives viser kostnaden ved å være forsiktig. Accuracy rapporteres også, men er ikke hovedmål siden feilene ikke er like alvorlige.

## 2. Baselines

In [ ]:
from sklearn.dummy import DummyClassifier

results = []

# 1) Majority class: predikerer alltid den vanligste klassen i treningsdata
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
results.append(report("Majority class", yte, dummy.predict(Xte)))

# 2) Enkel regel: alt med lukt er giftig
results.append(report("Regel: odor != n", yte, (Xte["odor"] != "n").astype(int)))

pd.DataFrame(results)

,Modell,Recall (giftig),Precision (giftig),Accuracy,False negatives,False positives
0,Majority class,0.000,0.000,0.518,783,0
1,Regel: odor != n,0.971,0.823,0.886,23,163


In [ ]:
mask = (Xte["odor"] != "n") & (yte == 0)       # spiselige sopper som regelen kaller giftige
print(Xte.loc[mask, "odor"].value_counts())    # hvilke lukter er det?
print("Totalt:", mask.sum())

odor
a    82
l    81
Name: count, dtype: int64
Totalt: 163


**Majority class** predikerer alltid spiselig: recall 0 og accuracy xx. Accuracy ser altså akseptabel ut selv om ingen giftige sopper fanges opp.

**Regel "odor != n"** (alt med lukt er giftig): recall 0,971, precision 0,823 og accuracy 0.518.
- 23 false negatives: giftige sopper uten lukt.
- 163 false positives: alle fra lukttypene `a` (82) og `l` (81).

Lukt alene er ikke et trygt kriterium. Baselinene er sammenligningsgrunnlaget for modellene i kap. 3.